# TA City — Phase 0 evaluation + baseline anchor (R125–R128)

Evaluates the seed-controlled calibration batch rasters and builds the skill
anchor against the three ABM-generated null baselines:

| Step | Cell | What it does |
|---|---|---|
| 1 | CONFIG | paths, scenario→run mapping, tolerances |
| 2 | helpers | ESRI ASCII reader, run-raster discovery, ensemble aggregation |
| 3 | references | urban_early/late → `obs_new`, valid mask |
| 4 | run rasters | R125–R128/seedNN final-tick exports (results vs baselines) |
| 5 | evaluate | `evaluate_scenario_fom` per seed, results & baselines separately |
| 6 | ensemble | per-seed rows → per-scenario MEANS (skill protocol, both sides) |
| 7 | anchor | `normalize_against_baselines` → skill vs strongest baseline per metric |
| 8 | export | fom/baseline/skill + per-seed CSVs into docs/ |

Conventions (project-agreed): all module arguments explicit keyword
arguments; `per_tick='final'`, `variant='hits_only'`, `cell_km=0.150`,
`start_year=2000`; ensemble means on both sides of the normalization.

Run *Restart & Run All* before saving; each cell is independently
re-runnable for troubleshooting (downstream cells depend on the `fom_*`
variables, not on printed state).

In [7]:
# 1. CONFIG ------------------------------------------------------------
from pathlib import Path

CONFIG = {
    "root": Path("../"),                       # path containing model/JAK and docs/
    "case_city": "JAK",
    "demand": 29656,                          # frozen gross target
    "final_tick": 25,                         # 2000-2025, one tick per year
    "calibration_years": [2000, 2025],        # years to calibrate
    "cell_km": 0.150,
    "tolerances": (0, 1, 2, 3, 5),
    "variant": "hits_only",                   # 'hits_only' | 'full'
    "scenario_run": {"s0": 125},              # results runs
    "scenario_baseline": {                     # baseline runs (nulls)
        "s_random": 126, "s_edge": 127, "s_roadbound": 128},
    "seeds": (11, 22, 33),
}

jak   = CONFIG["root"] / "model" / "JAK"
docs  = CONFIG["root"] / "docs"
out_root = jak / "data" / "out"
in_dir   = jak / "data" / "in"


In [ ]:
# 2a. imports + validation module ---------------------------------------
import re
import sys
import numpy as np
import pandas as pd

# module import: adjust path if running from another folder
sys.path.insert(0, str((CONFIG["root"] / "notebooks").resolve()))
import ti_city_10_calibration_fuzzy_fom_validation_module as fom
from ti_city_10_calibration_fuzzy_fom_validation_module import (
    observed_new_from_rasters, restrict_mask,
    evaluate_scenario_fom, normalize_against_baselines)
print("module loaded:", fom.__name__)
git_path = Path(r'C:\\GIT\\ti-city-model') # adjust this to match the path on your computer, e.g. 'D:\\GIT\\ti-city-model' or 'C:\\Users\\x\\GIT\\ti-city-model'

In [3]:
# 2b. helpers: ESRI ASCII reader -----------------------------------------
def read_ascii_grid(path):
    """Read an ESRI ASCII grid (ncols/nrows/xllcorner/yllcorner/cellsize/
    NODATA_value header) into (array, header-dict). Row 0 = top/north."""
    path = Path(path)
    lines = path.read_text().splitlines()
    header = {}
    i = 0
    while i < len(lines) and len(header) < 6:
        parts = lines[i].split()
        if len(parts) == 2 and parts[0].lower() in (
                "ncols", "nrows", "xllcorner", "yllcorner", "cellsize",
                "nodata_value"):
            header[parts[0].lower()] = parts[1]
            i += 1
        else:
            break
    arr = np.loadtxt(lines[i:], dtype=float)
    nrows, ncols = int(header["nrows"]), int(header["ncols"])
    assert arr.shape == (nrows, ncols), f"{path.name}: {arr.shape} != ({nrows},{ncols})"
    return arr, header

# 2c. helpers: run-raster discovery + loading ----------------------------
def read_run_raster(run_dir, final_tick):
    """Find the final-tick export in a run folder. Preference: income-status
    exports (classes 1-4 new urban, 5 initial urban), then any expansion_*."""
    pattern = re.compile(r"t(\d+)\.asc$", re.IGNORECASE)
    candidates = [(int(m.group(1)), p) for p in Path(run_dir).glob("*.asc")
                  if (m := pattern.search(p.name))]
    assert candidates, f"No tick-stamped .asc exports in {run_dir}"
    max_tick, _ = max(candidates, key=lambda c: c[0])
    if max_tick != final_tick:
        print(f"WARNING: {run_dir} final export is t{max_tick:03d}, expected t{final_tick:03d}")
    same_tick = [p for t, p in candidates if t == max_tick]
    income = [p for p in same_tick if "income" in p.name.lower()]
    chosen = income[0] if income else same_tick[0]
    arr, _ = read_ascii_grid(chosen)
    print(f"  loaded {chosen.name}  <- {Path(run_dir).name}")
    return arr

def load_run_dicts(mapping, label):
    """Build {scenario: {seed_key: array}} from the R-folder/seed structure."""
    results_dict = {}
    for scenario, run_num in mapping.items():
        series = {}
        for seed in CONFIG["seeds"]:
            run_dir = out_root / f"R{run_num}" / f"seed{seed}"
            series[f"seed{seed}"] = read_run_raster(run_dir, CONFIG["final_tick"])
        results_dict[scenario] = series
    print(f"loaded {label}: {', '.join(results_dict)}")
    return results_dict

def aggregate_ensemble(fom_table):
    """Per-seed rows -> per-scenario ensemble MEANS (skill protocol)."""
    metric_cols = [c for c in fom_table.columns
                   if c.startswith("FuzzyFoM")] + ["FoM", "N_sim",
                   "Hits", "Misses", "FalseAlarms"]
    metric_cols = [c for c in metric_cols if c in fom_table.columns]
    agg = fom_table.groupby("Scenario")[metric_cols].mean().reset_index()
    n = fom_table.groupby("Scenario").size().rename("n_seeds").reset_index()
    return agg.merge(n, on="Scenario")

In [4]:
# 3. reference layers: obs_new, valid mask ------------------------------
urban_early, h_early = read_ascii_grid(in_dir / "urban_early.asc")
urban_late,  _       = read_ascii_grid(in_dir / "urban_late.asc")
nodata = float(h_early.get("nodata_value", "-9999"))
valid_mask = (urban_early != nodata) & (urban_late != nodata)
urban_early_m = np.where(urban_early == nodata, 0, urban_early)
urban_late_m  = np.where(urban_late == nodata, 0, urban_late)
initial_urban = urban_early_m > 0

obs_all = observed_new_from_rasters(urban_late = urban_late_m,
                                    urban_early = urban_early_m)
obs_new = restrict_mask(obs_all, initial_urban = initial_urban,
                        valid_mask = valid_mask)
n_obs = int(np.sum(obs_new))
print(f"observed new-urban cells (AOI-masked): {n_obs:,}")
if abs(n_obs - CONFIG["demand"]) > CONFIG["demand"] * 0.01:
    print(f"WARNING: observed gross {n_obs:,} differs from frozen demand "
          f"{CONFIG['demand']:,} by >1% - check demand provenance.")

observed new-urban cells (AOI-masked): 29,656


In [ ]:
# 4. run rasters: results and baselines ---------------------------------
print("loading run rasters (final tick)...")
results_dict   = load_run_dicts(
    mapping = CONFIG["scenario_run"], 
    label = "results")
baseline_dict  = load_run_dicts(
    mapping = CONFIG["scenario_baseline"],
    label = "baselines")

loading run rasters (final tick)...
  loaded expansion_income_t025.asc  <- seed11
  loaded expansion_income_t025.asc  <- seed22
  loaded expansion_income_t025.asc  <- seed33
loaded results: s0
  loaded expansion_income_t025.asc  <- seed11
  loaded expansion_income_t025.asc  <- seed22
  loaded expansion_income_t025.asc  <- seed33
  loaded expansion_income_t025.asc  <- seed11
  loaded expansion_income_t025.asc  <- seed22
  loaded expansion_income_t025.asc  <- seed33
  loaded expansion_income_t025.asc  <- seed11
  loaded expansion_income_t025.asc  <- seed22
  loaded expansion_income_t025.asc  <- seed33
loaded baselines: s_random, s_edge, s_roadbound


In [ ]:
def load_run_dicts(mapping, label):
    """Build {scenario: {seed_key: array}} from the R-folder/seed structure."""
    results_dict = {}
    for scenario, run_num in mapping.items():
        series = {}
        for seed in CONFIG["seeds"]:
            run_dir = out_root / f"R{run_num}" / f"seed{seed}"
            series[f"seed{seed}"] = read_run_raster(run_dir, CONFIG["final_tick"])
        results_dict[scenario] = series
    print(f"loaded {label}: {', '.join(results_dict)}")
    return results_dict

In [ ]:
# 5. evaluate per seed: results and baselines SEPARATELY ---------------
fom_results_seeds = evaluate_scenario_fom(
    results_dict = results_dict,
    obs_new = obs_new,
    seeds_list = CONFIG["seeds"],
    initial_urban = initial_urban,
    valid_mask = valid_mask,
    tolerances = CONFIG["tolerances"],
    variant = CONFIG["variant"],
    per_tick = "final",
    start_year = CONFIG["calibration_years"][0],
    case_city = CONFIG["case_city"],
    cell_km = CONFIG["cell_km"],
    silent = False)

fom_baseline_seeds = evaluate_scenario_fom(
    results_dict = baseline_dict,
    obs_new = obs_new,
    seeds_list = CONFIG["seeds"],
    initial_urban = initial_urban,
    valid_mask = valid_mask,
    tolerances = CONFIG["tolerances"],
    variant = CONFIG["variant"],
    per_tick = "final",
    start_year = CONFIG["calibration_years"][0],
    case_city = CONFIG["case_city"],
    cell_km = CONFIG["cell_km"],
    silent = False)

Evaluating against 29,656 observed new-urban cells (excl. JAK initial urban & invalid cells).


ValueError: invalid literal for int() with base 10: 'seed33'

In [ ]:
# 6. ensemble MEANS on both sides (skill protocol) ----------------------
fom_results  = aggregate_ensemble(fom_results_seeds)
fom_baseline  = aggregate_ensemble(fom_baseline_seeds)
print("ensemble means:")
print(fom_baseline[["Scenario", "n_seeds", "N_sim", "FoM"]].to_string(index=False))

In [ ]:
# 7. skill anchor vs strongest baseline per metric ---------------- -------
fom_skill = normalize_against_baselines(
    fom_table = fom_results,
    baseline_fom_table = fom_baseline,
    perfect = 1.0)

In [ ]:
# 8a. anchor summary -----------------------------------------------------
fuzzy_cols = [c for c in fom_baseline.columns if c.startswith("FuzzyFoM")]
print("=== BASELINE ANCHOR (ensemble means over seeds 11/22/33) ===")
print(fom_baseline[["Scenario", "N_sim", "FoM"] + fuzzy_cols].to_string(index=False))

skill_cols = ["Scenario", "FoM"] + fuzzy_cols[:2] + \
             [c for c in fom_skill.columns if c.endswith("_skill")]
skill_cols = [c for c in skill_cols if c in fom_skill.columns]
print("\n=== MODEL (s0) + SKILL ===")
print(fom_skill[skill_cols].to_string(index=False))
print("\nReading: skill > 0 means s0 beats EVERY baseline on that metric "
      "(ensemble means); identify the binding baseline per metric before Morris.")

In [ ]:
# 8b. export to docs/ ----------------------------------------------------
exports = {
    "JAK_TA-model_fom_seeds_R125-128.csv": fom_results_seeds,
    "JAK_TA-model_baseline_fom_seeds_R125-128.csv": fom_baseline_seeds,
    "JAK_TA-model_fom_table_R125-128.csv": fom_results,
    "JAK_TA-model_baseline_fom_table_R125-128.csv": fom_baseline,
    "JAK_TA-model_skill_anchor_R125-128.csv": fom_skill,
}
for name, table in exports.items():
    table.to_csv(docs / name, index = False)
    print(f"written: docs/{name}")